# LLM vs ML: classifying agent skills (RQ3)

GPT, Gemini and DeepSeek classify the same skills as the ML models (`normal` / `suspicious` / `malicious`). Their
scores, tokens and cost are compared with the ML models on **exactly the same skills**.

| File | What it does |
|---|---|
| `llm_eval/llm_config.py` | **all settings**: the LLMs (model, API key name, prices), dataset choice, sample size |
| `llm_eval/data.py` | loads the exported skills, stratified sample |
| `llm_eval/prompt.py` | the prompt (identical for every LLM) and reading the answers |
| `llm_eval/providers.py` | one class per API (OpenAI, Gemini, OpenAI-compatible = DeepSeek) |
| `llm_eval/runner.py` | sends the skills, collects the answers, records tokens and cost |
| `llm_eval/evaluate.py` | scores LLMs and ML models on the same skills |

**Before the first run**
1. Run `skill_vulnerability_classifier_with_flags_v2.ipynb` up to Section 6, then **Section 10** (writes `llm_eval/data/`).
2. Copy `.env.example` to `.env` and paste your three API keys (the file is git-ignored).
3. Check the model IDs and prices in `llm_eval/llm_config.py`, then set `PRICES_VERIFIED = True`.

**How the batches work.** GPT and Gemini use the providers' Batch APIs (half price, results within 24 hours).
DeepSeek has no batch API, so its requests are sent in parallel and finish in minutes. Everything is saved to
`results/llm/<run>/<llm>/` as it arrives. Re-running a cell continues where it stopped: answered skills are skipped
and a batch is never submitted twice. You can close this notebook while the batches run.

In [1]:
%pip install -q -r requirements-llm.txt

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 1. Settings

`DATASET = "sample"` (default) classifies a stratified sample of `SAMPLE_SIZE` skills; `"full"` classifies all of them.
`SAMPLE_FROM = "test"` (default) draws from the ML test split (1,100 skills the ML models never saw). Change these here
or in `llm_config.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
import os

from llm_eval import data, evaluate, runner
from llm_eval import llm_config as cfg

cfg.DATASET = "sample"        # "sample" (stratified) or "full"
cfg.SAMPLE_SIZE = 1000
cfg.SAMPLE_FROM = "test"      # "test" (fair comparison with ML) or "all" (5,498 skills; ~80% were ML training data)
LLMS_TO_RUN = list(cfg.LLMS)  # e.g. ["gpt"] to run one LLM only

runner.load_env()
missing = [c["api_key_env"] for n, c in cfg.LLMS.items() if n in LLMS_TO_RUN and not __import__("os").environ.get(c["api_key_env"])]
print("API keys missing:", missing or "none")
pd.DataFrame({n: {"provider": c["provider"], "model": c["model"], **{f"$/1M {k}": v for k, v in c["prices"].items()},
                  "batch discount": c.get("batch_discount", 1.0)} for n, c in cfg.LLMS.items()}).T

## 2. The skills to classify

The sample keeps the share of every label **and** of every attack pattern inside each label (`STRATIFY_BY`).
The tables show how close the sample is to the pool it was drawn from (difference in percentage points).
The chosen skill IDs are saved to `selected_skills.csv`, so every LLM gets the same skills.

In [44]:
run = runner.Run()
print(f"Run '{run.name}': {len(run.skills):,} skills, {run.skills['truncated'].sum()} longer than "
      f"{cfg.MAX_INPUT_CHARS:,} characters (cut in the middle)\n")
pool, _ = data.load_export()
if cfg.SAMPLE_FROM == "test":
    pool = pool[pool["split"] == "test"]
display(data.representativeness(pool, run.skills, "judgment"))
rep = data.representativeness(pool, run.skills, "primary_pattern")
print(f"Attack patterns: {len(rep)} groups, largest difference {rep['difference'].abs().max():.2f} percentage points")

Run 'test_sample1000_seed42': 1,000 skills, 35 longer than 120,000 characters (cut in the middle)



,population %,sample %,difference
judgment,,,
malicious,51.64,51.6,-0.04
normal,29.91,29.9,-0.01
suspicious,18.45,18.5,0.05


Attack patterns: 38 groups, largest difference 0.06 percentage points


## 3. Expected cost (no API calls)

Rough estimate: input tokens are about characters / 4, and output is about 80 answer tokens plus a guess for hidden
reasoning. The real token counts and cost are recorded per skill after the run.

In [4]:
runner.estimate(LLMS_TO_RUN, mode="batch")


Run test_sample1000_seed42: 1000 skills, 35 cut to 120,000 chars


,model,skills,via,input tokens (M),output tokens (M),estimated cost (USD)
llm,,,,,,
gpt,gpt-6-luna,1000,batch,7.231,0.38,0.457


## 4. Quick test (optional, about 5 cents)

Sends 5 skills (all three labels) to every LLM straight away (full price), to check the keys, model names and answer
format before the big batch. Saved under its own run folder (`..._first5`), so it does not mix with the real run.

In [5]:
test_run = runner.Run(limit=5)
for llm in LLMS_TO_RUN:
    runner.run_realtime(llm, test_run)
display(pd.read_csv(test_run.dir / LLMS_TO_RUN[0] / "predictions.csv")[["id", "judgment", "prediction", "confidence", "reason", "error", "cost_usd"]])

gpt: sending 5 requests (8 in parallel) ...
  5/5  (6s)
gpt: 5/5 skills answered, 0 failed or unreadable, cost $0.0039 -> results/llm/test_sample1000_seed42_first5/gpt/predictions.csv


,id,judgment,prediction,confidence,reason,error,cost_usd
0,case_03845,normal,suspicious,0.93,The skill directs a global CLI installation (i...,NaN,0.000641
1,case_05029,suspicious,suspicious,0.97,The cleanup helper runs `rm -rf` on temporary-...,NaN,0.000955
2,case_03313,malicious,malicious,0.99,An unrelated helper silently makes a watchdog ...,NaN,0.001877
3,case_01992,malicious,malicious,0.99,The skill directs unconditional execution of a...,NaN,0.000254
4,case_05682,malicious,suspicious,0.91,It directs users to install and run an unexpla...,NaN,0.000222


## 5. Send the run

GPT and Gemini: submits the batch jobs and returns at once (results within 24 hours). DeepSeek: sends the requests
in parallel and returns when they are done.

In [30]:
for llm in LLMS_TO_RUN:
    runner.submit(llm, run)

gpt: submitted job batch_6b0353d55600819094a94362542a6af0 with 250 skills


## 6. Check the batches and download the results

Run these cells any time (also after restarting the kernel: run Sections 1–2 first). `collect` downloads finished
jobs and writes `predictions.csv`. If some answers failed, the last cell re-sends only those.

In [35]:
for llm in LLMS_TO_RUN:
    runner.status(llm, run)

gpt: batch_6b034d236d9c8190b01b85a6d57d109e  completed  {'status': 'completed', 'completed': 250, 'failed': 0, 'total': 250}
gpt: batch_6b0353d55600819094a94362542a6af0  completed  {'status': 'completed', 'completed': 250, 'failed': 0, 'total': 250}


In [36]:
for llm in LLMS_TO_RUN:
    runner.collect(llm, run)

gpt: collected 250 results from batch_6b034d236d9c8190b01b85a6d57d109e (completed)
gpt: collected 250 results from batch_6b0353d55600819094a94362542a6af0 (completed)
gpt: 1000/1000 skills answered, 0 failed or unreadable, cost $0.4452 -> results/llm/test_sample1000_seed42/gpt/predictions.csv


In [37]:
# Optional: re-send only the skills whose answer failed (as a new batch for GPT / Gemini), then run `collect` again
for llm in LLMS_TO_RUN:
    runner.submit(llm, run, retry_failed=True)

gpt: nothing to submit (0 skills already in unfinished jobs)


In [50]:
for llm in LLMS_TO_RUN:
    runner.build_predictions(llm, run)

gpt: 1000/1000 skills answered, 0 failed or unreadable, cost $0.4452 -> results/llm/test_sample1000_seed42/gpt/predictions.csv


## 7. LLMs vs the stacked model

The LLMs are compared with the **stacked model (all 6)** of Section 6.5 of the ML notebook: all 6 ML models as
level 0, a logistic regression on top. Set `SHOW_SINGLE_MODELS = True` to also show the 6 single ML models.

All models are scored on the same skills, with the same metrics as Section 6 of the ML notebook. A skill whose LLM
answer failed counts as wrong. Saved to `results/llm/<run>/comparison_llm_vs_ml.csv` (plus classification reports and
confusion matrices).

The stack's predictions come from Section 10 of the ML notebook: run Section 6.5 and then Section 10 there first.

In [ ]:
STACK = "Stack (all 6)"      # the stacked model from Section 6.5 of the ML notebook
SHOW_SINGLE_MODELS = False   # True = also show the 6 single ML models (DT, NB, SVM, KNN, RF, XGBoost)

ml_meta = json.loads(cfg.DATA_META.read_text())
COMPARE_WITH = [STACK] + (ml_meta["ml_models"] if SHOW_SINGLE_MODELS else [])
table, reports, cms = evaluate.compare(run.name, ml_models=COMPARE_WITH)
cols = ["type", "model", "macro F1", "macro precision", "macro recall", "accuracy", "F1 normal", "F1 suspicious",
        "F1 malicious", "failed answers", "cost (USD)", "cost per 1k skills (USD)", "est. cost all 5,498 skills (USD)",
        "input tokens", "output tokens", "reasoning tokens", "via"]
display(table[[c for c in cols if c in table.columns]].round(4))

In [ ]:
FIG_DIR = run.dir / "figures"
FIG_DIR.mkdir(exist_ok=True)
TYPE_COLORS = {"ML": "#2a78d6", "LLM": "#eb6834"}     # same palette as the ML notebook
INK, MUTED, GRID = "#1f1f1f", "#6b6b6b", "#e6e6e6"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 10,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "axes.edgecolor": GRID, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "legend.frameon": False})

# Macro F1 of every model (highest at the top)
t = table.sort_values("macro F1")
kind = t["type"].str.split().str[0]
fig, ax = plt.subplots(figsize=(8, 0.42 * len(t) + 1.2))
ax.barh(t.index, t["macro F1"].astype(float), color=kind.map(TYPE_COLORS), height=0.6)
for y, v in enumerate(t["macro F1"].astype(float)):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9, color=INK)
ax.set_xlim(0, 1.08); ax.grid(axis="y", visible=False)
ax.set_xlabel("macro F1 (same skills for every model)")
vs = "ML models" if SHOW_SINGLE_MODELS else STACK
ax.set_title(f"LLMs vs {vs}: macro F1 on {int(t['skills scored'].iloc[0])} skills")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in TYPE_COLORS.values()], labels=list(TYPE_COLORS),
          loc="lower right")   # ML = the stack (and the single models if shown)
fig.savefig(FIG_DIR / "01_macro_f1_llm_vs_ml.png"); plt.show()

# Cost of the LLMs (a separate chart: different unit, never a second y-axis)
llm_t = table[table["type"] == "LLM"].sort_values("cost per 1k skills (USD)")
fig, ax = plt.subplots(figsize=(8, 0.42 * len(llm_t) + 1.2))
ax.barh(llm_t.index, llm_t["cost per 1k skills (USD)"].astype(float), color=TYPE_COLORS["LLM"], height=0.6)
for y, v in enumerate(llm_t["cost per 1k skills (USD)"].astype(float)):
    ax.text(v, y, f"  ${v:.2f}", va="center", fontsize=9, color=INK)
ax.grid(axis="y", visible=False); ax.set_xlabel("US$ per 1,000 skills (as run: batch price where used)")
ax.set_title("LLM cost per 1,000 skills")
ax.set_xlim(0, llm_t["cost per 1k skills (USD)"].astype(float).max() * 1.25)
fig.savefig(FIG_DIR / "02_llm_cost_per_1k.png"); plt.show()

In [ ]:
# Confusion matrices (share of each true class), LLMs first, then the stacked model
BLUES = LinearSegmentedColormap.from_list("blues", ["#f7fbff", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
show = [n for n in table.index if n.startswith("LLM")] + [f"ML: {STACK}"]
fig, axes = plt.subplots(1, len(show), figsize=(4.6 * len(show), 4.2), squeeze=False)
for ax, name in zip(axes[0], show):
    cm = cms[name].values
    pct = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(pct, cmap=BLUES, vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]}\n{pct[i, j]:.0%}", ha="center", va="center", fontsize=9,
                    color="white" if pct[i, j] > 0.55 else INK)
    ax.set_xticks(range(3)); ax.set_xticklabels(evaluate.LABELS); ax.set_yticks(range(3))
    ax.set_yticklabels(evaluate.LABELS if ax is axes[0][0] else [])   # class names once, on the left
    ax.set_xlabel("predicted"); ax.grid(False); ax.set_title(name)
axes[0][0].set_ylabel("true")
# failed LLM answers ("unparsed") are not a column here, so a row can add up to less than 100%
fig.savefig(FIG_DIR / "03_confusion_llm_vs_stack.png"); plt.show()

In [41]:
for name, rep in reports.items():
    print(f"==================== {name} ====================")
    print(rep)

==================== LLM: deepseek ====================
              precision    recall  f1-score   support

      normal      0.924     0.930     0.927       299
  suspicious      0.753     0.724     0.738       185
   malicious      0.944     0.953     0.949       516

    accuracy                          0.904      1000
   macro avg      0.874     0.869     0.871      1000
weighted avg      0.903     0.904     0.903      1000

==================== LLM: gemini ====================
              precision    recall  f1-score   support

      normal      0.894     0.987     0.938       299
  suspicious      0.938     0.730     0.821       185
   malicious      0.968     0.986     0.977       516

    accuracy                          0.939      1000
   macro avg      0.933     0.901     0.912      1000
weighted avg      0.940     0.939     0.936      1000

==================== LLM: gpt ====================
              precision    recall  f1-score   support

      normal      0.91

## 8. What the LLMs got wrong

The skills an LLM misclassified, with its one-sentence reason. Useful for the error analysis in the write-up.

In [54]:
for llm in LLMS_TO_RUN:
    p = pd.read_csv(run.dir / llm / "predictions.csv")
    wrong = p[p["prediction"] != p["judgment"]]
    print(f"{llm}: {len(wrong)} wrong of {len(p)}")
    display(wrong[["id", "judgment", "prediction", "confidence", "reason", "truncated"]].head(10))

gpt: 179 wrong of 1000


,id,judgment,prediction,confidence,reason,truncated
2,case_03004,suspicious,malicious,0.99,"The skill bundles hard-coded live database, St...",False
5,case_00957,normal,suspicious,0.93,"The backup purpose is explicit, but the script...",False
7,case_04481,malicious,suspicious,0.91,The stock-price queries use relevant finance d...,False
15,case_04058,normal,suspicious,0.91,The converter uploads selected documents to a ...,False
19,case_00560,normal,suspicious,0.91,It explicitly enables stealth scraping and Clo...,False
25,case_04980,normal,suspicious,0.94,"The example plugin records prompts, messages, ...",False
26,case_01540,suspicious,malicious,0.90,It uses “highest priority” instructions to ove...,False
34,case_05350,normal,suspicious,0.98,The messaging helper constructs a command stri...,False
35,case_05264,suspicious,normal,0.92,The scripts perform the disclosed web-scraping...,False
36,case_04825,suspicious,malicious,0.98,"The detector returns hard-coded, fabricated sa...",False
